# Prueba C — Número de Interferencias

Barrido de 1, 2 y 4 fuentes de interferencia simultáneas.

**Cómo ejecutar:** correr la sección *Setup* una vez por sesión de Colab, luego la celda de *Ejecución*.

## Setup — ejecutar una vez por sesión de Colab
Montar Drive, clonar el repo, instalar dependencias y actualizar el código.

In [ ]:
# Import the drive module from Google Colab
from google.colab import drive

# Mount Google Drive to the virtual machine
drive.mount('/content/drive')


Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [ ]:
# 3. Descargar tu código temporalmente
%cd /content
!git clone https://github.com/MatiasVereert/Vision-Aided-Beamformer.git

/content
fatal: destination path 'Vision-Aided-Beamformer' already exists and is not an empty directory.


In [ ]:
# Install standard audio and evaluation libraries from PyPI
!pip install noisereduce mir_eval

# Install pysepm directly from its official GitHub repository
!pip install git+https://github.com/schmiph2/pysepm.git

# Install pb_bss directly from the Paderborn University GitHub repository
!pip install git+https://github.com/fgnt/pb_bss.git

# h5py and pyarrow are usually pre-installed, but we can ensure they are up to date
!pip install h5py pyarrow
!pip install git+https://github.com/LCAV/pyroomacoustics.git

!pip install git+https://github.com/fgnt/nara_wpe.git

!pip install paderbox

!pip install ai_edge_litert

  Cloning https://github.com/schmiph2/pysepm.git to /tmp/pip-req-build-7umzzdca
  Running command git clone --filter=blob:none --quiet https://github.com/schmiph2/pysepm.git /tmp/pip-req-build-7umzzdca
  Resolved https://github.com/schmiph2/pysepm.git to commit 7ef88aff2c56201a2d0470aaeb58e77e47a914d2
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
  Using cached https://github.com/ludlows/python-pesq/archive/master.zip
  Preparing metadata (setup.py) ... done
  Using cached https://github.com/jfsantos/SRMRpy/archive/master.zip
  Preparing metadata (setup.py) ... done
  Using cached https://github.com/detly/gammatone/archive/master.zip
  Preparing metadata (setup.py) ... done
  Cloning https://github.com/fgnt/pb_bss.git to /tmp/pip-req-build-exodlcnt
  Running command git clone --filter=blob:none --quiet https://github.com/fgnt/pb_bss.git /tmp/pip-req-build-exodlcnt
  Resolved https://github.com/fgnt

In [ ]:
%cd /content/Vision-Aided-Beamformer
!git pull origin main

/content/Vision-Aided-Beamformer
From https://github.com/MatiasVereert/Vision-Aided-Beamformer
 * branch            main       -> FETCH_HEAD
Already up to date.


## Generación de combinaciones de interferencia

In [ ]:
import itertools

# Base parameters for positions and audio files
posiciones = [(-30, 1.0), (45, 1.0), (-60, 1.0), (75, 1.0)]
audios = [0, 1, 2, 3]

todas_las_interferencias = []

# Iterate for 1, 2, and 4 simultaneous sources (explicitly skipping 3)
for n_fuentes in [1, 2, 4]:

    # 1. Choose which 'n' physical positions to use (no repetition, order doesn't matter)
    for combo_pos in itertools.combinations(posiciones, n_fuentes):

        # 2. For those 'n' positions, assign all permutations of distinct audios
        for combo_aud in itertools.permutations(audios, n_fuentes):

            # 3. Pair each position with its corresponding audio
            config_actual = []
            for i in range(n_fuentes):
                angulo = combo_pos[i][0]
                distancia = combo_pos[i][1]
                id_audio = combo_aud[i]
                config_actual.append((angulo, distancia, id_audio))

            todas_las_interferencias.append(config_actual)

print(f"[*] Total de combinaciones de interferencia generadas: {len(todas_las_interferencias)}")

[*] Total de combinaciones de interferencia generadas: 112


## Ejecución

In [ ]:
import sys
import os
import numpy as np
import shutil
from datetime import datetime


# 1. Definición de rutas del repositorio en Colab
repo_root = '/content/Vision-Aided-Beamformer'
src_path = os.path.join(repo_root, 'src')

# Añadir rutas al sys.path para importaciones absolutas del paquete
if repo_root not in sys.path:
    sys.path.append(repo_root)
if src_path not in sys.path:
    sys.path.append(src_path)

# Mover el directorio de trabajo a src para dependencias relativas internas
%cd {src_path}

try:
    import tensorflow as tf
    TFLITE_AVAILABLE = True
except ImportError:
    print("[!] TensorFlow no detectado. Los módulos neuronales DTLN se desactivarán.")
    TFLITE_AVAILABLE = False

# Importar el orquestador refactorizado y los wrappers de los beamformers
from evaluation.full_benchmark_test_dtln_mird import run_mird_grid_search
from evaluation.bf_wrappers import (
    DS_Processor,
    MVDR_Recursive_Processor,
    DTLN_MB_MVDR_SOUDEN_Processor,
    ORACLE_MB_MVDR_SOUDEN_Processor,
)

from propagation.mird_loader import MirdDatasetProvider, generate_mird_linear_array

# 2. Inicialización de los intérpretes DTLN en Colab
# Apuntamos a la carpeta donde residen los modelos .tflite dentro de tu repo
model_1_path = os.path.join(repo_root, "src/dnn_denoise/models/model_quant_1.tflite")
model_2_path = os.path.join(repo_root, "src/dnn_denoise/models/model_quant_2.tflite")

interpreter_1, interpreter_2 = None, None
if TFLITE_AVAILABLE and os.path.exists(model_1_path) and os.path.exists(model_2_path):
    try:
        interpreter_1 = tf.lite.Interpreter(model_path=model_1_path)
        interpreter_1.allocate_tensors()

        interpreter_2 = tf.lite.Interpreter(model_path=model_2_path)
        interpreter_2.allocate_tensors()
        print("[*] Modelos DTLN TFLite cargados y alojados correctamente en memoria Colab.")
    except Exception as e:
        print(f"[!] Falló la reserva de memoria para DTLN: {e}. Se procederá sin mejora neuronal.")
        interpreter_1, interpreter_2 = None, None
else:
    print("[*] Ejecutando evaluación en modo estrictamente Acústico/Espacial (Sin DTLN).")

# Ajusta la ruta si la carpeta está dentro de otra subcarpeta en tu Drive
input_dir = "/content/drive/MyDrive/Benchmarks_tesis/inputs"

# Ajustar la ruta de mird dataset
mird_dir = "/content/drive/MyDrive/Benchmarks_tesis/rirs"

provider = MirdDatasetProvider(root_dir = mird_dir)

# ===== Config WPE (fija; actualizar con el optimo hallado en E0) =====
WPE_TAPS  = 10   # <-- reemplazar con taps* de E0
WPE_DELAY = 3    # <-- reemplazar con delay* de E0

base_config = {
    'fs': 16000,
    'duration': 20,
    't_early': 0.050,  # (50 ms)
    'array_center': [3.0, 3.0, 1.2], # Virtual translation anchor for SimAcoustic
    'mird_spacing': "3-3-3-8-3-3-3", # Target linear array spacing in the dataset

    'snr_db': 60.0,
    'source_path': os.path.join(input_dir, "p002_emo_adoration_sentences.wav"),
    'interf_paths': [
        os.path.join(input_dir, "hairdryer_07_SH_MKH800.wav"),
        os.path.join(input_dir, "flute_music.wav"),
        os.path.join(input_dir, "drill_07_RHODE_NT1.wav"),
        os.path.join(input_dir, "techno_gated commune.wav"),

    ],

    'wpe_taps': WPE_TAPS,
    'wpe_delay': WPE_DELAY,
    'wpe_alpha': 0.9999,
    'wpe_stft_size': 512,
    'wpe_stft_shift': 128,

    'stft_window': 512,
    'stft_overlap': 384,

    'dtln_model_path': os.path.join(repo_root, "src/dnn_denoise/models/model_quant_1.tflite"),
    'eval_references': ['anechoic', 'early', 'reverberant']

    }

# Physical parameters adapted to MIRD boundaries
param_grid = {
    'rt60': [0.610],
    'target_angle': [0],
    'target_dist': [1.0],
    'interf_configs': todas_las_interferencias,
    # Sin Errores
    'isir_db': [ 0],
    'mismatch_gain': [0],
    'mismatch_phase': [0],
    'use_wpe': [True],
    'error_angle_deg': [0.0],
    'error_distance_m': [0.0]
}
processors_dict = {
    "DS":          DS_Processor(nperseg=512, noverlap=384),
    "MVDR-Geo":    MVDR_Recursive_Processor(nperseg=512, noverlap=384, min_loading=1e-6),
    "NM-MVDR":     DTLN_MB_MVDR_SOUDEN_Processor(min_loading=1e-9, alpha=0.99),
    "Oracle-MVDR": ORACLE_MB_MVDR_SOUDEN_Processor(min_loading=1e-9, alpha=0.99),
}

# --- EJECUCIÓN SECUENCIAL ---
print("\n" + "="*60)
print("INICIANDO EXPERIMENTO C: NÚMERO DE INTERFERENCIAS")
print("="*60)

# Almacenamiento temporal en el disco local de la máquina virtual (SSD rápido)
RUN_TAG = datetime.now().strftime("%Y%m%d_%H%M")
temp_output_dir = f"/content/results_temp/wpe_Exp_C_{RUN_TAG}"
# Destino final persistente montado en Google Drive
drive_output_dir = f"/content/drive/MyDrive/Tesis_Beamformers/results/wpe_Exp_C_{RUN_TAG}"

os.makedirs(temp_output_dir, exist_ok=True)
os.makedirs(drive_output_dir, exist_ok=True)

# Ejecutar el Grid Search apuntando la salida pesada al disco local
df_C = run_mird_grid_search(
    grid_params=param_grid,
    dataset_provider =provider,
    processors=processors_dict,
    scene_base_config=base_config,
    output_dir=temp_output_dir,
    interpreter_1=interpreter_1,  # Inyectamos instancias TFLite si existen
    interpreter_2=interpreter_2
)

print("\n[INFO] Procesamiento en Colab finalizado exitosamente.")
print("[INFO] Sincronizando catálogo HDF5 y Parquet hacia Google Drive...")

# Transferencia segura recursiva para preservar toda la data en Drive
shutil.copytree(temp_output_dir, drive_output_dir, dirs_exist_ok=True)

print("\n[ÉXITO] Experimento A completado y respaldado de forma segura en Google Drive.")

## Descarga para análisis local

In [ ]:
# Colab no puede escribir directo en el repo local. Este helper baja el CSV de
# metricas a tu carpeta de Descargas; luego movelo a:
#   /home/matias/Documents/Tesis/Vision-Aided-Beamformer/data_analysis
from google.colab import files
_csv = os.path.join(drive_output_dir, "mird_benchmark_metrics.csv")
_dl  = f"{os.path.basename(drive_output_dir)}_metrics.csv"
shutil.copy(_csv, f"/content/{_dl}")
files.download(f"/content/{_dl}")
